# **NLP** - tabiiy tilga ishlov berish

 `NLP` (Natural Language Processing) - bu tabiiy tilga ishlov berish degan ma'noni anglatadi. Bu soha sun'iy intellektning bir qismi bo'lib, kompyuterlarga inson tilini tushunish, talqin qilish va generatsiya qilish imkonini beradi. Oddiy qilib aytganda, kompyuterlar bizning tilimizda gaplasha olishini, yozishimizni tushunishini va hatto o'zi ham matn yaratishini ta'minlaydi.

Asosiy vazifalari:

Matnni tushunish: Kompyuter matnning ma'nosini anglay olishi. Masalan, sentiment analizi (matnning ijobiy yoki salbiy ekanligini aniqlash).
Matn yaratish: Kompyuterning o'zi yangi matnlar, xabarlar yoki hisobotlar yozishi.

Tarjima: Bir tildan boshqa tilga avtomatik tarjima qilish.
Nutqni tanib olish: Ovozli buyruqlarni matnga aylantirish.
Savol-javob tizimlari: Foydalanuvchilarning savollariga javob beradigan botlar yaratish.

Bu soha juda ko'p joyda ishlatiladi, masalan, spamlarni filtrlash, chat-botlar, nutq yordamchilari (Siri, Google Assistant), avtomatik tarjimalar va boshqalar.



///////////////////////

In [ ]:
!wget https://github.com/laxmimerit/IMDB-Movie-Reviews-Large-Dataset-50k/raw/master/train.xlsx

--2026-03-10 04:56:52--  https://github.com/laxmimerit/IMDB-Movie-Reviews-Large-Dataset-50k/raw/master/train.xlsx
Resolving github.com (github.com)... 140.82.116.4
Connecting to github.com (github.com)|140.82.116.4|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://raw.githubusercontent.com/laxmimerit/IMDB-Movie-Reviews-Large-Dataset-50k/master/train.xlsx [following]
--2026-03-10 04:56:52--  https://raw.githubusercontent.com/laxmimerit/IMDB-Movie-Reviews-Large-Dataset-50k/master/train.xlsx
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.111.133, 185.199.108.133, 185.199.109.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.111.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 13723193 (13M) [application/octet-stream]
Saving to: ‘train.xlsx’

train.xlsx          100%[===================>]  13.09M  --.-KB/s    in 0.09s   

2026-03-10 04:56:53 (139 MB/s) - ‘train.x

###Yuklash

### Tozalash, (old ishlov berish)

###Model

###Dataset class

###train/test

###Bu jarayonlar ketma ketlikda bajariladi

In [ ]:
import pandas as pd
import numpy as np

url = '/content/train.xlsx'
df = pd.read_excel(url)
df.head()

,Reviews,Sentiment
0,"When I first tuned in on this morning news, I ...",neg
1,"Mere thoughts of ""Going Overboard"" (aka ""Babes...",neg
2,Why does this movie fall WELL below standards?...,neg
3,Wow and I thought that any Steven Segal movie ...,neg
4,"The story is seen before, but that does'n matt...",neg


### DataFrame, 25000 qator va 2 ustundan iborat. U asosan kinolar sharhi (review) va ularning sentimenti (ijobiy yoki salbiy) haqidagi ma'lumotlarni o'z ichiga oladi. Ustun nomlari:

### `reviews`: Kinoga berilgan yozma sharh.
### `Sentiment`: Sharhning ijobiy (positive) yoki salbiy (negative) ekanligini ko'rsatadi.

In [ ]:
df.shape

(25000, 2)

In [ ]:
df = df.sample(500, random_state=42)
df

# Tasodifiy 500 qator tanlab olindi dastlabki misol tariqasida va tezroq ishlashi uchun

,Reviews,Sentiment
6868,I really think that people are taking the wron...,pos
24016,There was a stylish approach to this film on t...,pos
9668,My wife and I just finished watching Bûsu AKA ...,neg
13640,As with all environmentally aware films from t...,pos
14018,"With an absolutely amazing cast and crew, this...",neg
...,...,...
9774,"Who would think Andy Griffith's ""Helen Crump"" ...",pos
4068,this has got to be one of those films where th...,neg
15582,Made me think about it for days after seeing i...,pos
18379,"Well, I get used after awhile to read comments...",pos


In [ ]:
df = df.sample(500, random_state=42).reset_index(drop=True)
df

# Bu 500 qator indexi har hil edi va biz reset_index orqali ularni yana qaytadan tartibli raqamladik

,Reviews,Sentiment
0,This movie is just great. It's entertaining fr...,pos
1,"Oh, Sam Mraovich, we know you tried so hard. T...",neg
2,Tony Goldwyn is a good actor who evidently is ...,neg
3,I don't watch soaps. My grandmother still watc...,pos
4,'The Last Wave' is far more than the sum of it...,pos
...,...,...
495,Family problems abound in real life and that i...,pos
496,My wife and I found this film to be highly uns...,neg
497,You spend most of this two-hour film wondering...,pos
498,I was given the opportunity to see this 1926 f...,pos


In [ ]:
df.shape

(500, 2)

///////////////////////

In [ ]:
import torch
from collections import Counter
# torch chuqur o'rganish uchun, Counter esa elementlarning chastotasini hisoblash uchun ishlatiladi.

def tokenize(text):
  return text.lower().split()
# Berilgan matnni kichik harflarga o'tkazadi (lower()) va so'zlarga ajratadi (split()).
# Misol uchun, "Hello World" -> ['hello', 'world'].

counter = Counter(word for text in df['Reviews'] for word in tokenize(text))
# df['reviews'] ustunidagi barcha sharhlardagi so'zlarning necha marta uchraganini
# hisoblab chiqadi. Bu kod barcha so'zlarni yig'ib, ularning sonini sanaydi.
most_common = counter.most_common(2000) # Eng ko'p uchraydigan 2000 so'zni va ularning sonini oladi
vocab = {word: i+2 for i, (word, _) in enumerate(most_common)}
vocab['<pad>'] = 0
vocab['<unk>'] = 1
# Tanlangan 2000 ta eng ko'p uchraydigan so'zlar uchun lug'at (vocabulary) yaratadi.
# Har bir so'zga noyob raqamli indeks beradi. <pad> (to'ldirish) uchun 0 va <unk> (noma'lum so'zlar)
# uchun 1 indekslari qo'shiladi. Bu NLP modellarida matnni raqamli ko'rinishga o'tkazishda muhim.

def encode(text):
  return torch.tensor([vocab.get(word, 1) for word in tokenize(text)])
# encode(text) funksiyasi: Berilgan matndagi so'zlarni yuqorida yaratilgan vocab lug'ati
# yordamida raqamli indekslarga o'tkazadi. Agar so'z lug'atda bo'lmasa, unga <unk> (1)
# indeksi beriladi. Bu matnni modelga kiritishdan oldin "raqamlashtirish" jarayonidir.

In [ ]:
print(most_common)
# buyrug'i faqat eng ko'p uchraydigan so'zlarni va ularning sharhlarda
# necha marta takrorlanganligini (chastotasini) ko'rsatadi.
# Faqat 2000 ta so'z uchun, counter esa mavjud barcha so'zlarni necha marta takrorlangani aks ettiradi

[('the', 6716), ('a', 3192), ('and', 3155), ('of', 2855), ('to', 2695), ('is', 2161), ('in', 1743), ('i', 1427), ('this', 1381), ('that', 1372), ('it', 1349), ('/><br', 1007), ('was', 957), ('as', 922), ('for', 884), ('with', 791), ('but', 773), ('his', 609), ('film', 602), ('on', 591), ('are', 589), ('you', 587), ('movie', 577), ('not', 563), ('have', 562), ('he', 533), ('be', 491), ('from', 463), ('at', 443), ('they', 442), ('one', 427), ('by', 424), ('all', 408), ('an', 407), ('like', 382), ('just', 377), ('who', 377), ('so', 376), ('or', 364), ('about', 354), ('has', 331), ('some', 309), ('what', 305), ("it's", 302), ('if', 290), ('her', 286), ('-', 283), ('more', 277), ('when', 262), ('which', 259), ('would', 257), ('my', 252), ('very', 248), ('even', 245), ('only', 242), ('there', 241), ('out', 241), ('had', 234), ('their', 225), ('really', 219), ('than', 218), ('good', 216), ('up', 211), ('no', 208), ('she', 202), ('see', 200), ('can', 199), ('were', 193), ('been', 189), ('becau

In [ ]:
list(vocab.items())[:10]
# Bunda har bir so'zga berilgan noyob indexlari bilan aks ettirildi yani tanlangan 2000 ta so'zlar uchun

[('the', 2),
 ('a', 3),
 ('and', 4),
 ('of', 5),
 ('to', 6),
 ('is', 7),
 ('in', 8),
 ('i', 9),
 ('this', 10),
 ('that', 11)]

///////////////////////

In [ ]:
from torch.utils.data import Dataset, DataLoader
# PyTorch kutubxonasidan Dataset va DataLoader sinflarini import qiladi. Dataset o'zingizning
# ma'lumotlaringizni PyTorch formatiga moslash uchun asosiy sinf bo'lsa, DataLoader ma'lumotlarni
# modelga kichik "paketlar" (batch) ko'rinishida yetkazib berish uchun ishlatiladi.

class IMDBDataset(Dataset):
  def __init__(self, df):
    self.X = [encode(text) for text in df['Reviews']]
    self.y = [torch.tensor(1.0 if s == 'pos' else 0.0) for s in df['Sentiment']]

  def __len__(self):
    return len(self.X)

  def __getitem__(self, idx):
    return self.X[idx], self.y[idx]

# class IMDBDataset(Dataset):: Bu IMDBDataset nomli yangi sinfni aniqlaydi. U Dataset sinfidan meros oladi,
# ya'ni PyTorch tomonidan tan olinadigan maxsus ma'lumotlar to'plami hisoblanadi.

# def __init__(self, df):: Bu sinfning konstruktori bo'lib, IMDBDataset obyekti yaratilganda
# avtomatik ravishda ishga tushadi. U df (DataFrame) qabul qiladi:

# self.X = [encode(text) for text in df['Reviews']]: Bu qator df dagi har bir kino sharhini (Reviews ustuni)
# encode funksiyasi yordamida raqamli indekslarga aylantiradi (ya'ni, so'zlarni raqamlarga o'giradi) va natijalarni
# self.X ro'yxatiga saqlaydi. self.X datasetdagi kirish ma'lumotlari bo'ladi.
# self.y = [torch.tensor(1.0 if s == 'pos' else 0.0) for s in df['Sentiment']]: Bu qator df dagi har bir sharhning
# sentimentini (Sentiment ustuni, 'pos' yoki 'neg') PyTorch tensorlariga aylantiradi. 'pos' (positive) 1.0 ga, 'neg'
# (negative) 0.0 ga teng bo'ladi. self.y datasetdagi chiqish ma'lumotlari (yoki maqsad vektorlari) bo'ladi.
# def __len__(self):: Bu metod datasetdagi umumiy elementlar sonini (ya'ni, sharhlar sonini) qaytarishi shart.

# return len(self.X): Datasetdagi elementlar sonini self.X ro'yxatining uzunligi orqali hisoblaydi va qaytaradi.
# def __getitem__(self, idx):: Bu metod berilgan idx (indeks) bo'yicha datasetdan bitta elementni (sharh va uning sentimentini) qaytarishi shart.

# return self.X[idx], self.y[idx]: idx-chi sharhning raqamli ko'rinishini (self.X[idx]) va uning sentimentini
# (self.y[idx]) qaytaradi. Bu PyTorch DataLoader uchun har bir ma'lumotni olish usuli.


In [ ]:
imdb_dataset = IMDBDataset(df)
print(imdb_dataset.X[:2]) # Dastlabki 2 ta sharhning kodlangan ko'rinishini ko'rsatish

[tensor([  10,   24,    7,   37,    1,   45,  496,   29,  523,    6,    2,  775,
         406,  193,    1,   28,   30,    2, 1086,    5,  118, 1977,  524,    2,
         369,  122,    8,   53,  917,   10,   24,    7,  708,    1,   33,    2,
           1,   13,    1,    1,  918,   73,    2,  275,    5,    2,  101,    1,
           1,  287,    1,    1,  338,   17,  110,  709,    1,  133,    3,   90,
        1087,   15,    1,    1,    4,  666,  379,    1,   32,    5,    2,  120,
         370,  205,  103,    5,  110,   15,    3,    1,   13,  667,    2,  194,
          11,    1,    5,    2,   24,  239,  347,   30,    3,    1,   17,    3,
         246,  173,    5,   96,  354,   21,   12,  133,  104,   16,    3,  173,
           5,    1,  174,   13,    1, 1699]), tensor([1480, 1978,    1,   74,  115,   23,  587,   39,    1,   10,    7,  118,
           1,    1,    3,    1,  710,    6,    2,  313,    5,  165,   11,   23,
          22,  321,  267,    1,   73,    2, 1979,    5, 1700,  432,    1,

In [ ]:
def collate_fn(batch):
  Xs, ys = zip(*batch)
  max_len = max(len(x) for x in Xs)
  # Her bir x ni (review) nol bilan to'ldirib, bir xil uzunlikka keltirish
  padded_Xs = [torch.cat([x, torch.zeros(max_len - len(x), dtype=x.dtype)], dim=0) for x in Xs]
  return torch.stack(padded_Xs), torch.stack(ys)

train_loader = DataLoader(IMDBDataset(df), batch_size=32, shuffle=True, collate_fn=collate_fn)

# def collate_fn(batch):: Bu collate_fn nomli funksiyani aniqlaydi. Bu funksiya PyTorch DataLoader tomonidan har bir ma'lumotlar
# batchini (guruhini) tayyorlash uchun ishlatiladi. batch o'zida bir nechta (bizning holatda 32 ta) (input, label) juftliklarini saqlaydi.
# Xs, ys = zip(*batch): Bu qator batch ichidagi ma'lumotlarni ikkita alohida ro'yxatga ajratadi: Xs (kirish ma'lumotlari,
# ya'ni kodlangan sharhlar) va ys (chiqish ma'lumotlari, ya'ni sentiment yorliqlari).
# max_len = max(len(x) for x in Xs): Joriy batch ichidagi barcha sharhlardan eng uzunining uzunligini aniqlaydi.
# Bu, sharhlarni bir xil uzunlikka keltirish uchun kerak bo'ladi.
# padded_Xs = [torch.cat([x, torch.zeros(max_len - len(x), dtype=x.dtype)], dim=0) for x in Xs]: Bu qator har bir sharhni (x) eng uzun sharh
# (max_len) uzunligiga moslab, kerakli joylarga nollar (zeros) bilan to'ldiradi (padding). Bu, har xil uzunlikdagi kirish matnlarini neyron
# tarmoqqa berishdan oldin bir xil o'lchamga keltirish uchun muhim.
# return torch.stack(padded_Xs), torch.stack(ys): To'ldirilgan sharhlarni (padded_Xs) va sentiment yorliqlarini (ys) alohida torch.Tensor
# ob'ektlariga aylantirib qaytaradi. torch.stack bir nechta kichik tensorlarni bitta katta tensorga birlashtiradi.
# train_loader = DataLoader(IMDBDataset(df), batch_size=32, shuffle=True, collate_fn=collate_fn): Bu qator PyTorch DataLoader obyektini yaratadi.
# Bu obyekt ma'lumotlarni modelga batchlar shaklida yetkazib berish uchun javobgardir.
# IMDBDataset(df): Bizning yaratgan IMDBDataset klassimizdan ma'lumotlar to'plamini oladi.
# batch_size=32: Har bir batchda 32 ta misol (sharh va uning sentimenti) bo'lishini bildiradi.
# shuffle=True: Har bir epoch boshida ma'lumotlar to'plamini aralashtirishni ta'minlaydi, bu modelning o'zlashtirishini yaxshilaydi.
# collate_fn=collate_fn: DataLoaderga har bir batchni yuklashdan oldin yuqorida tushuntirilgan collate_fn funksiyasini ishlatishni aytadi.
# Bu, turli uzunlikdagi sharhlarni to'g'ri ishlashga yordam beradi.

///////////////////////

In [ ]:
import torch.nn as nn

class SentimentRNN(nn.Module):
  def __init__(self, vocab_size, embedding_dim=64, hidden_dim=128):
    super().__init__()
    self.embedding = nn.Embedding(vocab_size, embedding_dim)
    self.rnn = nn.RNN(embedding_dim, hidden_dim, batch_first=True)
    self.fc = nn.Linear(hidden_dim, 1)
    self.sigmoid = nn.Sigmoid()
# class SentimentRNN(nn.Module):: SentimentRNN nomli klassni aniqlaydi, bu PyTorchdagi neyron tarmoqlar uchun asosiy nn.Module klassidan meros oladi.
# def __init__(self, vocab_size, embedding_dim=64, hidden_dim=128):: Modelning konstruktori:
# self.embedding = nn.Embedding(...): Har bir so'zni raqamli indeksdan zich vektorga (embedding) aylantiradi.
# vocab_size lug'at hajmi, embedding_dim esa vektorning o'lchami.
# self.rnn = nn.RNN(...): Rekurrent Neyron Tarmoq (RNN) qatlami. Matn ketma-ketliklarini qayta ishlash uchun javobgar.
# embedding_dim kirish vektori o'lchami, hidden_dim yashirin holat o'lchami.
# self.fc = nn.Linear(hidden_dim, 1): To'liq bog'langan (Fully Connected) qatlam. RNNning oxirgi yashirin holatini bitta chiqish qiymatiga o'tkazadi.
# self.sigmoid = nn.Sigmoid(): Sigmoid aktivatsiya funksiyasi. Chiqish qiymatini 0 va 1 oralig'idagi ehtimollikka (sentimentga) aylantiradi.

  def forward(self, x):
    x = self.embedding(x)
    _, h = self.rnn(x)
    x = self.fc(h.squeeze(0))
    return self.sigmoid(x)
# def forward(self, x):: Modelning to'g'ridan-to'g'ri hisoblash (forward pass) qismi:
# x = self.embedding(x): Kirish x (so'z indekslari) embedding qatlami orqali vektorlarga aylantiriladi.
# _, h = self.rnn(x): Embedding vektorlari RNN qatlamiga kiritiladi. h bu oxirgi qadamdagi yashirin holatni ifodalaydi.
# x = self.fc(h.squeeze(0)): Yashirin holat (h) bir qatorli (linear) qatlam orqali o'tkaziladi. squeeze(0) batch o'lchamini olib tashlaydi.
# return self.sigmoid(x): Oxirgi natija Sigmoid funksiyasidan o'tkazilib, 0 dan 1 gacha bo'lgan sentiment ehtimolini qaytaradi.

In [ ]:
len(vocab)
# bu lug'atdagi (vocabulary) so'zlar sonini qaytaradi, ya'ni model ishlatadigan barcha noyob so'zlar va
# qo'shimcha maxsus tokenlar (<pad>, <unk>) sonini ko'rsatadi.

2002

In [ ]:
model = SentimentRNN(len(vocab))
criterion = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# model = SentimentRNN(len(vocab)): SentimentRNN klassidan yangi model obyektini yaratadi.
# len(vocab) parametri lug'atdagi so'zlar sonini ko'rsatadi, bu embedding qatlami uchun muhim.
# criterion = nn.BCELoss(): Binary Cross-Entropy Loss funksiyasini aniqlaydi. Bu ikkilik tasniflash muammolari
# (masalan, ijobiy/salbiy sentiment) uchun umumiy tanlov bo'lib, modelning bashoratlari va haqiqiy qiymatlar o'rtasidagi farqni o'lchaydi.
# optimizer = torch.optim.Adam(model.parameters(), lr=0.001): Adam optimizatorini sozlaydi. Bu optimizator modelning
# vaznlarini (parameters) o'quv jarayonida (learning rate lr=0.001 bilan) yangilash orqali loss funksiyasini minimallashtirish uchun ishlatiladi.



///////////////////////

In [ ]:
for epoch in range(10):
  total_loss = 0

  for X, y in train_loader:
    y_pred = model(X)
    # `y` ni `y_pred` bilan bir xil o'lchamga keltirish
    loss = criterion(y_pred, y.unsqueeze(1))

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    total_loss += loss.item()

  print(f"Epoch {epoch+1}, Loss: {total_loss:.4f}")

# for epoch in range(10):: Model 10 marta ma'lumotlar to'plami ustida o'qitiladi. Har bir aylanish 'epoch' deb ataladi.
# total_loss = 0: Har bir epoch boshida umumiy xato (loss) nolga o'rnatiladi.
# for X, y in train_loader:: train_loader yordamida ma'lumotlar kichik guruhlarga (X - kirish, y - chiqish/label) bo'lib yuklanadi.
# y_pred = model(X): Modelga kirish ma'lumotlari (X) beriladi va model bashorat qilgan natija (y_pred) olinadi.
# loss = criterion(y_pred, y.unsqueeze(1)): Bashorat qilingan natija (y_pred) va haqiqiy natija (y) orasidagi xato (farq)
# BCELoss funksiyasi yordamida hisoblanadi. y.unsqueeze(1) y ning o'lchamini y_predga moslashtiradi.
# optimizer.zero_grad(): Oldingi iteratsiyalardagi gradentlar tozalanadi.
# loss.backward(): Hisoblangan xato asosida modelning parametrlari bo'yicha gradentlar (parametrlarni qaysi yo'nalishda o'zgartirish kerakligi) hisoblanadi.
# optimizer.step(): Gradentlar yordamida modelning vaznlari (parametrlar) yangilanadi (o'qitiladi).
# total_loss += loss.item(): Joriy batchning xatosi umumiy xatoga qo'shiladi.
# print(f"Epoch {epoch+1}, Loss: {total_loss:.4f}"): Har bir epoch yakunida, o'sha epochdagi umumiy xato chop etiladi.
# Bu modelning o'qitish jarayonini kuzatishga yordam beradi.


Epoch 1, Loss: 11.2593
Epoch 2, Loss: 11.0688
Epoch 3, Loss: 11.2102
Epoch 4, Loss: 11.2013
Epoch 5, Loss: 11.0290
Epoch 6, Loss: 11.0183
Epoch 7, Loss: 11.0211
Epoch 8, Loss: 10.9385
Epoch 9, Loss: 10.8411
Epoch 10, Loss: 11.7034


In [ ]:
def predict(text):
  model.eval()
  with torch.no_grad():
    x = encode(text)
    x = x[:100]
    if len(x) < 100:
      x = torch.cat([x, torch.zeros(100-len(x), dtype=torch.long)])
    x = x.unsqueeze(0)
    y_pred = model(x)

    prob = y_pred.item()
    label = "positive" if prob > 0.5 else "negative"
    print(f"Ishonchi: {prob:.2f}, Bashorat: {label}")

# def predict(text):: Bu predict nomli funksiyani aniqlaydi. U matnni (text) kirish sifatida qabul qiladi va uning sentimentini (ijobiy yoki salbiy) bashorat qiladi.
# model.eval(): Modelni baholash rejimiga o'tkazadi. Bu, o'qitish paytida ishlatiladigan ba'zi qatlamlarni (masalan, Dropout) o'chirib qo'yadi.
# with torch.no_grad():: Bu blok ichida gradient hisoblashni o'chiradi. Bu bashorat qilish (inference) paytida xotira va hisoblash tezligini tejaydi, chunki biz vaznlarni yangilashga hojat yo'q.
# x = encode(text): Kirish matnini (text) oldindan aniqlangan encode funksiyasi yordamida raqamli indekslarga aylantiradi.
# x = x[:100]: Kodlangan matnning uzunligini 100 tagacha qisqartiradi, ya'ni faqat birinchi 100 ta so'z/tokenni oladi.
# if len(x) < 100:: Agar matn 100 ta so'zdan kam bo'lsa, bu shart tekshiriladi.
# x = torch.cat([x, torch.zeros(100-len(x), dtype=torch.long)]): Agar matn 100 ta so'zdan kam bo'lsa, uni 100 ta so'z uzunligiga yetkazish uchun oxiriga nollar bilan to'ldiradi (padding).
# x = x.unsqueeze(0): Kirish tensoriga batch o'lchamini qo'shadi. Model odatda batchlarda ma'lumot qabul qilganligi sababli, bitta kirishni [1, sequence_length] shaklida [batch_size, sequence_length] ga o'tkazadi.
# y_pred = model(x): Tayyorlangan kirish (x) modelga beriladi va modelning bashorati (y_pred) olinadi.
# prob = y_pred.item(): Modelning bashorat qilingan qiymatini PyTorch tensoridan oddiy Python raqamiga (float) aylantiradi. Bu ehtimollik (prob) bo'lib, 0 va 1 oralig'ida bo'ladi.
# label = "positive" if prob > 0.5 else "negative": Agar bashorat qilingan ehtimollik 0.5 dan katta bo'lsa, sentimentni "positive" (ijobiy), aks holda "negative" (salbiy) deb belgilaydi.
# print(f"Ishonchi: {prob:.2f}, Bashorat: {label}"): Bashorat qilingan ehtimollikni (ikki xona aniqlikda) va sentiment yorlig'ini chop etadi.

In [ ]:
predict("I love this movie")
predict("I hated this movie")
predict("I hated this movie")
predict("This movies very interesting")

Ishonchi: 0.85, Bashorat: positive
Ishonchi: 0.19, Bashorat: negative
Ishonchi: 0.19, Bashorat: negative
Ishonchi: 0.45, Bashorat: negative


## Barcha sharhlarni olmaganligimiz sababli bashorat deyarli yaxshi natija ko'rsatmadi